# US 수출 데이터 SARIMA 예측 및 DB 저장

이 노트북은 US 수출 데이터(expDlr)를 HS 코드별로 SARIMA 예측하고 MySQL DB에 저장합니다.

## 주요 기능
1. HS 코드별 월별 expDlr SARIMA 예측
2. 월별 → 분기별 자동 집계
3. 예측 파라미터 JSON 저장
4. created_at 기준 버전 관리
5. tqdm 진행바로 실시간 모니터링

## 1. 라이브러리 및 함수 정의

In [24]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from datetime import datetime
from typing import Optional, Dict, Tuple
import json

from statsmodels.tsa.statespace.sarimax import SARIMAX
from pandas.tseries.offsets import MonthEnd
from itertools import product
from sqlalchemy import create_engine, text
import pymysql
from tqdm import tqdm

print("라이브러리 로드 완료")

라이브러리 로드 완료


### 1.1 유틸리티 함수

In [25]:
def to_month_end(s):
    """날짜를 월말로 변환"""
    ts = pd.to_datetime(s)
    if isinstance(ts, pd.Timestamp):
        return ts + MonthEnd(0)
    return ts + MonthEnd(0)


def ensure_sorted_unique_dates(df: pd.DataFrame, date_col: str = "time") -> pd.DataFrame:
    """날짜 정렬 및 중복 제거"""
    d = df.copy()
    d[date_col] = pd.to_datetime(d[date_col])
    d[date_col] = d[date_col] + MonthEnd(0)
    return d.sort_values(date_col).drop_duplicates([date_col]).reset_index(drop=True)


print("유틸리티 함수 정의 완료")

유틸리티 함수 정의 완료


### 1.2 SARIMA 파라미터 탐색

In [26]:
def find_best_sarima_params(
    y_train: pd.Series,
    seasonal_period: int = 12,
    p_values=(0, 1, 2),
    d_values=(0, 1),
    q_values=(0, 1, 2),
    P_values=(0, 1),
    D_values=(0, 1),
    Q_values=(0, 1),
    ic: str = "aic",
    max_order_sum: int = 8,
) -> Tuple[tuple, tuple]:
    """최적 SARIMA 파라미터 탐색"""
    best_ic = np.inf
    best_order = (1, 1, 1)
    best_sorder = (1, 1, 0, seasonal_period)

    for p, d, q in product(p_values, d_values, q_values):
        for P, D, Q in product(P_values, D_values, Q_values):
            if (p + q + P + Q) > max_order_sum:
                continue
            order = (p, d, q)
            sorder = (P, D, Q, seasonal_period)
            try:
                m = SARIMAX(
                    y_train.astype(float),
                    order=order,
                    seasonal_order=sorder,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
                fit = m.fit(disp=False)
                val = fit.aic if ic.lower() == "aic" else fit.bic
                if np.isfinite(val) and val < best_ic:
                    best_ic, best_order, best_sorder = val, order, sorder
            except Exception:
                continue

    return best_order, best_sorder


print("파라미터 탐색 함수 정의 완료")

파라미터 탐색 함수 정의 완료


### 1.3 월별 SARIMA 예측

In [27]:
def run_sarima_forecast_monthly(
    df: pd.DataFrame,
    hs_code: str,
    forecast_months: int = 12,
    target_col: str = "expDlr",
    ic: str = "aic",
    min_obs: int = 24,
    date_col: str = "time"
):
    """
    월별 SARIMA 예측 수행

    Parameters:
    -----------
    df : DataFrame with columns ['time', 'expDlr']
    hs_code : HS 코드
    forecast_months : 예측할 개월 수
    target_col : 예측 대상 컬럼
    ic : 정보기준 ('aic' or 'bic')
    min_obs : 최소 관측치 수
    date_col : 날짜 컬럼명

    Returns:
    --------
    result_df : 예측 결과 포함 DataFrame
    metadata : 모델 정보 딕셔너리
    """
    results = {
        "hs_code": hs_code,
        "model": "SARIMA",
        "frequency": "monthly",
        "order": None,
        "seasonal_order": None,
        "aic": None,
        "bic": None,
        "error": None
    }

    try:
        # 날짜 컬럼 자동 감지
        if date_col not in df.columns:
            if 'date' in df.columns:
                date_col = 'date'
            elif 'time' in df.columns:
                date_col = 'time'
            else:
                raise ValueError("날짜 컬럼을 찾을 수 없습니다.")

        # 데이터 정렬 및 정제
        d = ensure_sorted_unique_dates(df[[date_col, target_col]], date_col)
        d = d.rename(columns={date_col: "date_month_end"})

        # 시계열 생성
        y = pd.Series(d[target_col].values, index=d["date_month_end"]).dropna()

        if len(y) < min_obs:
            results["error"] = f"관측치 부족: {len(y)} < {min_obs}"
            return d, results

        # 최적 파라미터 탐색
        order, sorder = find_best_sarima_params(
            y, seasonal_period=12, ic=ic
        )

        # 모델 적합
        model = SARIMAX(
            y,
            order=order,
            seasonal_order=sorder,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        fit = model.fit(disp=False)

        # 예측
        forecast = fit.forecast(steps=forecast_months)

        # 미래 날짜 생성
        last_date = y.index.max()
        future_dates = pd.date_range(
            last_date + MonthEnd(1),
            periods=forecast_months,
            freq="M"
        )

        # 결과 DataFrame 생성
        result_df = d.copy()
        result_df["expDlr_forecast"] = result_df[target_col]

        # 미래 예측값 추가
        for i, future_date in enumerate(future_dates):
            if future_date not in result_df["date_month_end"].values:
                new_row = pd.DataFrame({
                    "date_month_end": [future_date],
                    target_col: [np.nan],
                    "expDlr_forecast": [float(forecast.iloc[i])]
                })
                result_df = pd.concat([result_df, new_row], ignore_index=True)
            else:
                result_df.loc[
                    result_df["date_month_end"] == future_date,
                    "expDlr_forecast"
                ] = float(forecast.iloc[i])

        # 메타데이터 저장
        results["order"] = order
        results["seasonal_order"] = sorder
        results["aic"] = float(fit.aic)
        results["bic"] = float(fit.bic)

        # 정렬
        result_df = result_df.sort_values("date_month_end").reset_index(drop=True)

    except Exception as e:
        results["error"] = str(e)
        result_df = d.copy() if 'd' in locals() else df.copy()
        if "expDlr_forecast" not in result_df.columns:
            result_df["expDlr_forecast"] = result_df.get(target_col)

    return result_df, results


print("월별 예측 함수 정의 완료")

월별 예측 함수 정의 완료


### 1.4 분기별 집계

In [28]:
def aggregate_to_quarter(monthly_df: pd.DataFrame) -> pd.DataFrame:
    """
    월별 데이터를 분기별로 집계

    Parameters:
    -----------
    monthly_df : 월별 데이터 (date_month_end, expDlr, expDlr_forecast 컬럼 필요)

    Returns:
    --------
    quarterly_df : 분기별 집계 데이터
    """
    df = monthly_df.copy()
    df["quarter"] = df["date_month_end"].dt.to_period("Q")

    # 분기별 합계
    quarterly = df.groupby("quarter").agg({
        "expDlr": "sum",
        "expDlr_forecast": "sum"
    }).reset_index()

    # 분기 말일로 변환
    quarterly["date_quarter_end"] = quarterly["quarter"].dt.to_timestamp(how="end")
    quarterly["date_quarter_end"] = quarterly["date_quarter_end"] + MonthEnd(0)

    quarterly = quarterly.drop("quarter", axis=1)

    return quarterly


print("분기별 집계 함수 정의 완료")

분기별 집계 함수 정의 완료


### 1.5 DB 저장용 데이터 준비

In [29]:
def prepare_db_table_monthly(
    df: pd.DataFrame,
    hs_code: str,
    metadata: Dict,
    created_at: datetime
) -> pd.DataFrame:
    """
    월별 예측 결과를 DB 저장용 형태로 변환

    Columns: hs_code, date_month_end, expDlr, expDlr_forecast,
             is_forecast, params, created_at
    """
    db_df = df.copy()
    db_df["hs_code"] = hs_code
    db_df["is_forecast"] = db_df["expDlr"].isna().astype(int)

    # 파라미터를 JSON 문자열로 저장
    params_dict = {
        "model": metadata.get("model"),
        "order": metadata.get("order"),
        "seasonal_order": metadata.get("seasonal_order"),
        "aic": metadata.get("aic"),
        "bic": metadata.get("bic")
    }
    db_df["params"] = json.dumps(params_dict, ensure_ascii=False)
    db_df["created_at"] = created_at

    # 컬럼 순서 정리
    db_df = db_df[[
        "hs_code", "date_month_end", "expDlr", "expDlr_forecast",
        "is_forecast", "params", "created_at"
    ]]

    return db_df


def prepare_db_table_quarter(
    df: pd.DataFrame,
    hs_code: str,
    metadata: Dict,
    created_at: datetime
) -> pd.DataFrame:
    """
    분기별 예측 결과를 DB 저장용 형태로 변환

    Columns: hs_code, date_quarter_end, expDlr, expDlr_forecast,
             is_forecast, params, created_at
    """
    db_df = df.copy()
    db_df["hs_code"] = hs_code
    db_df["is_forecast"] = db_df["expDlr"].isna().astype(int)

    # 파라미터를 JSON 문자열로 저장
    params_dict = {
        "model": metadata.get("model"),
        "frequency": "quarterly",
        "order": metadata.get("order"),
        "seasonal_order": metadata.get("seasonal_order"),
        "aic": metadata.get("aic"),
        "bic": metadata.get("bic")
    }
    db_df["params"] = json.dumps(params_dict, ensure_ascii=False)
    db_df["created_at"] = created_at

    # 컬럼 순서 정리
    db_df = db_df[[
        "hs_code", "date_quarter_end", "expDlr", "expDlr_forecast",
        "is_forecast", "params", "created_at"
    ]]

    return db_df


print("DB 준비 함수 정의 완료")

DB 준비 함수 정의 완료


### 1.6 DB 저장 함수

In [30]:
def save_to_mysql(
    df: pd.DataFrame,
    table_name: str,
    db_info: Dict,
    if_exists: str = "append"
):
    """
    DataFrame을 MySQL DB에 저장

    Parameters:
    -----------
    df : 저장할 DataFrame
    table_name : 테이블 이름
    db_info : DB 연결 정보 {'host', 'user', 'password', 'database', 'port'}
    if_exists : 'append' or 'replace'
    """
    # SQLAlchemy 엔진 생성
    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
        f"{db_info['host']}:{db_info.get('port', 3306)}/{db_info['database']}",
        echo=False
    )

    try:
        # DataFrame을 MySQL에 저장
        df.to_sql(
            name=table_name,
            con=engine,
            if_exists=if_exists,
            index=False,
            chunksize=1000
        )
        print(f"저장 완료: {table_name} ({len(df):,} rows)")
    except Exception as e:
        print(f"DB 저장 실패: {str(e)}")
        raise
    finally:
        engine.dispose()


print("DB 저장 함수 정의 완료")

DB 저장 함수 정의 완료


### 1.7 전체 처리 메인 함수

In [31]:
def forecast_and_save_all_hs_codes(
    trade_data: pd.DataFrame,
    db_info: Dict,
    forecast_months: int = 12,
    min_obs: int = 24,
    ic: str = "aic",
    date_col: str = "time"
):
    """
    모든 HS 코드에 대해 예측 수행 및 DB 저장

    Parameters:
    -----------
    trade_data : US 무역 데이터 (columns: hs_code, time, expDlr)
    db_info : DB 연결 정보
    forecast_months : 예측 개월 수
    min_obs : 최소 관측치 수
    ic : 정보 기준
    date_col : 날짜 컬럼명 (기본: 'time', 'date'일 수도 있음)
    """
    import datetime as dt
    created_at = dt.datetime.now()

    # date_col이 존재하는지 확인하고 자동 조정
    if date_col not in trade_data.columns:
        if 'date' in trade_data.columns:
            date_col = 'date'
        elif 'time' in trade_data.columns:
            date_col = 'time'
        else:
            raise ValueError("날짜 컬럼을 찾을 수 없습니다. 'time' 또는 'date' 컬럼이 필요합니다.")

    # HS 코드 목록
    hs_codes = trade_data["hs_code"].unique()
    total_codes = len(hs_codes)

    print(f"총 {total_codes:,}개 HS 코드 처리 시작...")
    print(f"예측 개월 수: {forecast_months}")
    print(f"생성 시각: {created_at}")
    print("=" * 80)

    monthly_results = []
    quarter_results = []
    failed_codes = []

    # tqdm 진행바 사용
    pbar = tqdm(hs_codes, desc="예측 진행", unit="HS코드")

    for hs_code in pbar:
        try:
            # 진행 상태 업데이트
            pbar.set_postfix({"현재": str(hs_code)[:10], "성공": len(monthly_results), "실패": len(failed_codes)})

            # HS 코드별 데이터 추출
            hs_data = trade_data[trade_data["hs_code"] == hs_code].copy()

            # 날짜 컬럼명을 'time'으로 통일
            if date_col != 'time':
                hs_data = hs_data.rename(columns={date_col: 'time'})

            # 월별 예측
            monthly_df, metadata = run_sarima_forecast_monthly(
                df=hs_data,
                hs_code=hs_code,
                forecast_months=forecast_months,
                target_col="expDlr",
                ic=ic,
                min_obs=min_obs
            )

            if metadata.get("error"):
                failed_codes.append({"hs_code": hs_code, "error": metadata["error"]})
                continue

            # 분기별 집계
            quarter_df = aggregate_to_quarter(monthly_df)

            # DB 저장용 형태로 변환
            db_monthly = prepare_db_table_monthly(
                monthly_df, hs_code, metadata, created_at
            )
            db_quarter = prepare_db_table_quarter(
                quarter_df, hs_code, metadata, created_at
            )

            monthly_results.append(db_monthly)
            quarter_results.append(db_quarter)

        except Exception as e:
            failed_codes.append({"hs_code": hs_code, "error": str(e)})
            continue

    pbar.close()
    print("=" * 80)

    # 결과 요약
    print(f"\n처리 완료:")
    print(f"  성공: {len(monthly_results):,}개")
    print(f"  실패: {len(failed_codes):,}개")

    if failed_codes:
        print(f"\n실패한 HS 코드 샘플 (최대 10개):")
        for item in failed_codes[:10]:
            print(f"  - {item['hs_code']}: {item['error']}")

    # 모든 결과 통합 및 저장
    if monthly_results:
        all_monthly = pd.concat(monthly_results, ignore_index=True)
        print(f"\n월별 데이터: {len(all_monthly):,} rows")
        print("DB 저장 중...", end=" ")
        save_to_mysql(
            all_monthly,
            table_name="us_trade_export_monthly_with_forecast",
            db_info=db_info,
            if_exists="append"
        )
    else:
        print("\n월별 데이터 없음")

    if quarter_results:
        all_quarter = pd.concat(quarter_results, ignore_index=True)
        print(f"분기별 데이터: {len(all_quarter):,} rows")
        print("DB 저장 중...", end=" ")
        save_to_mysql(
            all_quarter,
            table_name="us_trade_export_quarter_with_forecast",
            db_info=db_info,
            if_exists="append"
        )
    else:
        print("분기별 데이터 없음")

    print("\n모든 처리 완료!")
    return created_at


print("메인 함수 정의 완료")

메인 함수 정의 완료


## 2. DB 연결 정보 설정

In [32]:
from DATA.stock_invest_function import *

# DB 정보 입력
db_info = {
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'host': get_db_host(),
    'port': '3307',
    'database': 'investar'
}

# 연결 테스트
try:
    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
        f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
    )
    with engine.connect() as conn:
        print("DB 연결 성공")
    engine.dispose()
except Exception as e:
    print(f"DB 연결 실패: {e}")

DB 연결 성공


## 3. 데이터 로드

In [33]:
# 옵션 1: CSV 파일에서 로드
# trade_df = pd.read_csv("us_trade_data.csv")

# 옵션 2: DB에서 로드
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
    f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

query = """
SELECT
    hs_code,
    date,
    expDlr
FROM us_trade_data
WHERE expDlr IS NOT NULL
ORDER BY hs_code, date
"""

trade_df = pd.read_sql(query, engine)
engine.dispose()

# 날짜 변환
trade_df['date'] = pd.to_datetime(trade_df['date'])

print(f"데이터 로드 완료: {len(trade_df):,} rows")
print(f"HS 코드 수: {trade_df['hs_code'].nunique():,}")
print(f"기간: {trade_df['date'].min()} ~ {trade_df['date'].max()}")

데이터 로드 완료: 68,994 rows
HS 코드 수: 478
기간: 2013-01-31 00:00:00 ~ 2025-09-30 00:00:00


## 4. 데이터 확인

In [34]:
# 기본 정보
print("데이터 구조:")
print(trade_df.head())

print("\n기초 통계:")
print(trade_df['expDlr'].describe())

# HS 코드별 관측치 수
hs_counts = trade_df.groupby('hs_code').size().sort_values(ascending=False)
print("\nHS 코드별 관측치 수 (상위 10개):")
print(hs_counts.head(10))

print(f"\n최소 관측치: {hs_counts.min()}")
print(f"최대 관측치: {hs_counts.max()}")
print(f"평균 관측치: {hs_counts.mean():.1f}")

데이터 구조:
  hs_code       date       expDlr
0  100199 2013-01-31  738003507.0
1  100199 2013-02-28  844245600.0
2  100199 2013-03-31  936336637.0
3  100199 2013-04-30  952592398.0
4  100199 2013-05-31  804767583.0

기초 통계:
count    6.899400e+04
mean     2.389889e+08
std      6.873778e+08
min      0.000000e+00
25%      5.754625e+07
50%      9.133350e+07
75%      1.707071e+08
max      1.425480e+10
Name: expDlr, dtype: float64

HS 코드별 관측치 수 (상위 10개):
hs_code
100199    153
853650    153
850110    153
848790    153
848690    153
848640    153
848620    153
848420    153
848390    153
848340    153
dtype: int64

최소 관측치: 44
최대 관측치: 153
평균 관측치: 144.3


## 5. 단일 HS 코드 테스트

In [35]:
# 테스트용 HS 코드 선택
test_hs = trade_df.groupby('hs_code').size().idxmax()
print(f"테스트 HS 코드: {test_hs}")
test_hs = '854231'
test_data = trade_df[trade_df['hs_code'] == test_hs].copy()
print(f"테스트 데이터: {len(test_data)} 관측치")
print(f"기간: {test_data['date'].min()} ~ {test_data['date'].max()}")

테스트 HS 코드: 100199
테스트 데이터: 153 관측치
기간: 2013-01-31 00:00:00 ~ 2025-09-30 00:00:00


In [36]:
# 월별 예측 수행
monthly_result, metadata = run_sarima_forecast_monthly(
    df=test_data,
    hs_code=test_hs,
    forecast_months=14,
    target_col='expDlr',
    ic='aic',
    min_obs=24
)

print("예측 완료!")
print(f"모델: SARIMA{metadata['order']}x{metadata['seasonal_order']}")
print(f"AIC: {metadata['aic']:.2f}")
print(f"BIC: {metadata['bic']:.2f}")

if metadata.get('error'):
    print(f"에러: {metadata['error']}")

예측 완료!
모델: SARIMA(1, 1, 2)x(0, 1, 1, 12)
AIC: 5180.04
BIC: 5194.18


In [37]:
# 예측 결과 확인
print("예측 결과 (최근 14개월):")
display_cols = ['date_month_end', 'expDlr', 'expDlr_forecast']
print(monthly_result[display_cols].tail(24))

예측 결과 (최근 14개월):
    date_month_end        expDlr  expDlr_forecast
143     2024-12-31  2.817561e+09     2.817561e+09
144     2025-01-31  3.278119e+09     3.278119e+09
145     2025-02-28  2.762375e+09     2.762375e+09
146     2025-03-31  2.872100e+09     2.872100e+09
147     2025-04-30  3.127032e+09     3.127032e+09
148     2025-05-31  2.816068e+09     2.816068e+09
149     2025-06-30  2.743521e+09     2.743521e+09
150     2025-07-31  2.901765e+09     2.901765e+09
151     2025-08-31  2.804615e+09     2.804615e+09
152     2025-09-30  2.643287e+09     2.643287e+09
153     2025-10-31           NaN     2.713161e+09
154     2025-11-30           NaN     2.626517e+09
155     2025-12-31           NaN     2.595130e+09
156     2026-01-31           NaN     2.777276e+09
157     2026-02-28           NaN     2.382464e+09
158     2026-03-31           NaN     2.578499e+09
159     2026-04-30           NaN     2.638785e+09
160     2026-05-31           NaN     2.664504e+09
161     2026-06-30           NaN 

## 6. 전체 HS 코드 예측 및 DB 저장

In [38]:
# 예측 파라미터 설정
FORECAST_MONTHS = 14  # 예측할 개월 수
MIN_OBS = 24          # 최소 관측치 수
IC = "aic"            # 정보 기준

print(f"예측 설정:")
print(f"  예측 개월: {FORECAST_MONTHS}")
print(f"  최소 관측치: {MIN_OBS}")
print(f"  정보 기준: {IC}")
print("\n실행하시겠습니까? (아래 셀 실행)")

예측 설정:
  예측 개월: 14
  최소 관측치: 24
  정보 기준: aic

실행하시겠습니까? (아래 셀 실행)


In [ ]:
# 실행
created_at = forecast_and_save_all_hs_codes(
    trade_data=trade_df,
    db_info=db_info,
    forecast_months=FORECAST_MONTHS,
    min_obs=MIN_OBS,
    ic=IC,
    date_col="date"  # 또는 "date"
)

print(f"\n저장 완료 시각: {created_at}")

총 478개 HS 코드 처리 시작...
예측 개월 수: 14
생성 시각: 2025-12-13 23:00:31.750711


예측 진행:   0%|          | 0/478 [00:00<?, ?HS코드/s, 현재=100199, 성공=0, 실패=0]

## 7. DB 저장 결과 확인

In [ ]:
# 월별 테이블 확인
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
    f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

query_monthly = f"""
SELECT *
FROM us_trade_export_monthly_with_forecast
WHERE created_at = '{created_at}'
LIMIT 100
"""

monthly_check = pd.read_sql(query_monthly, engine)
print("월별 테이블 저장 확인:")
print(f"저장된 행 수: {len(monthly_check):,}")
print(monthly_check.head(10))

In [ ]:
# 분기별 테이블 확인
query_quarter = f"""
SELECT *
FROM us_trade_export_quarter_with_forecast
WHERE created_at = '{created_at}'
LIMIT 100
"""

quarter_check = pd.read_sql(query_quarter, engine)
print("\n분기별 테이블 저장 확인:")
print(f"저장된 행 수: {len(quarter_check):,}")
print(quarter_check.head(10))

engine.dispose()

In [ ]:
# 파라미터 정보 확인
if len(monthly_check) > 0:
    sample_params = monthly_check.iloc[0]['params']
    print("\n저장된 파라미터 예시:")
    print(json.loads(sample_params))

## 8. 요약 통계

In [ ]:
# 전체 예측 통계
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
    f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

query_summary = f"""
SELECT
    COUNT(DISTINCT hs_code) as hs_code_count,
    COUNT(*) as total_rows,
    SUM(is_forecast) as forecast_rows,
    MIN(date_month_end) as min_date,
    MAX(date_month_end) as max_date
FROM us_trade_export_monthly_with_forecast
WHERE created_at = '{created_at}'
"""

summary = pd.read_sql(query_summary, engine)
print("예측 요약 통계:")
print(summary)

engine.dispose()